# HBN RestingState EEG Preprocessing

**Dataset:** `R1_mini_L100_bdf` (Healthy Brain Network mini release)
**System:** EGI GSN-HydroCel-129 · **Sampling rate:** 100 Hz · **Task:** `RestingState`
**Scope:** research-internship preprocessing pipeline (not a methods paper).

## Pipeline

```
Raw EEG
  -> 1-40 Hz band-pass (zero-phase IIR Butterworth)
  -> Variance-based bad-channel detection (robust / MAD)
  -> Bad-channel interpolation (spherical spline)
  -> Average reference
  -> Effective-rank check                  (single-subject validation only)
  -> ICA (rank-limited, manual rejection)  (single-subject validation only)
  -> Alpha-preservation validation         (single-subject validation only)
Batch stage: same pipeline up to epoching, WITHOUT ICA (components are subject-specific).
```

## Design notes grounded in the supplied literature

These are the explicit decisions a reviewer would probe, with the relevant
paper and the *kind* of response (documentation / small code adjustment /
major change). The pipeline deliberately stays simple; most concerns are
handled by documentation plus two small code adjustments.

- **1-40 Hz zero-phase IIR (Robbins et al. 2020).** MNE applies IIR filters
  with `filtfilt` (zero-phase; effective order doubled to ~8). Robbins et al.
  report the largest cross-pipeline disagreement in the low-frequency / delta
  band, driven by high-pass and baselining choices. *Response: documentation.*
  If delta is ever reported downstream, treat it as preprocessing-sensitive.

- **Bad-channel detection before referencing + interpolation (Kim et al. 2023;
  Robbins et al. 2020).** Detecting/handling bad channels *before* the average
  reference prevents a bad channel from contaminating the reference. Detection
  uses a robust (median/MAD) statistic on log-variance rather than mean +- 3*SD,
  because the mean/SD are themselves inflated by the outliers they are meant to
  find. *Response: small code adjustment.*

- **Average reference + spherical-spline interpolation reduce data rank
  (Kim et al. 2023).** Non-linear spline interpolation and average referencing
  each lower the effective rank; running ICA for more components than the rank
  produces "ghost ICs" -- flat-PSD, no-ERP components that nonetheless show
  *normal-looking* topographies and so survive visual inspection. The fix is to
  compute the effective rank and limit ICA to it (PCA-for-rank-adjustment).
  *Response: small code adjustment* (compute rank, pass `n_components=rank`).

- **ICA stays manual and is single-subject only.** Component indices are
  subject-specific; any exclusion list shown is an example for the validation
  subject only and is NOT reused in the batch stage. *Response: documentation +
  structure* (ICA excluded from batch).

- **Subject-level leakage (Brookshire et al. 2024).** Fixed-length epochs are
  saved one file per subject so any downstream model can split *by subject*
  (e.g. GroupKFold), never by epoch. Treating epochs -- or channels -- as
  independent observations inflates accuracy. *Response: documentation.*

- **Inter-recording amplitude scaling (Robbins et al. 2020).** Robust
  per-recording amplitude scaling reduces cross-recording variability; noted
  for any downstream cross-subject comparison but not applied here, since this
  is a preprocessing deliverable. *Response: documentation.*


> **Important -- what the saved data does and does not contain.**
>
> The batch stage in section 15 saves one `*_task-RestingState_clean_epo.fif`
> per subject. These files have been filtered (1-40 Hz Butterworth, zero-phase),
> had bad channels detected (robust MAD log-variance) and **interpolated**,
> re-referenced to the **average reference**, and epoched into 4-s
> non-overlapping windows.
>
> They have **not** been ICA-cleaned. ICA is fitted and inspected only on the
> single validation subject in sections 10-11; the batch loop in section 15
> deliberately omits ICA because artifact components are subject-specific and
> must be inspected per subject. Any downstream figure, table or report
> produced from `preprocessed_epochs/*.fif` must state this -- residual ocular,
> muscle and line-noise activity is still present in the saved data.

## 1. Configuration and dataset discovery

A single configuration cell (no duplicates). The dataset root is located
automatically; all tunable values are named constants with documented units so
nothing is buried as a magic number deeper in the notebook.

In [ ]:

import os
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------- Constants -----------------------------------
# Single source of truth. Changing a value here changes it everywhere,
# including the batch stage, so the validated and batched pipelines cannot
# silently diverge.

TASK            = "RestingState"      # BIDS task label
MONTAGE_NAME    = "GSN-HydroCel-129"  # EGI HydroCel net
EXPECTED_NCHAN  = 129                 # sanity check against wrong montage
SFREQ_EXPECTED  = 100.0               # Hz (documented; not enforced)

L_FREQ          = 1.0                 # Hz, high-pass edge
H_FREQ          = 40.0                # Hz, low-pass edge
IIR_ORDER       = 4                   # Butterworth order (zero-phase -> ~8 eff.)

# Robust bad-channel detection: flag channels whose log-variance is more than
# BAD_CH_MAD_THRESH robust-SDs (1.4826*MAD) from the median log-variance.
BAD_CH_MAD_THRESH = 5.0

EPOCH_DURATION  = 4.0                 # s, fixed-length epochs

# Effective-rank tolerance (Kim et al. 2023): keep eigenvalues larger than
# RANK_REL_TOL * (largest eigenvalue). Relative (not absolute 1e-7) so it is
# invariant to recording units (V vs uV).
RANK_REL_TOL    = 1e-6

# Posterior-electrode selection for alpha validation. MNE head coordinates are
# in metres, nasion-relative RAS: y < 0 is posterior. We take electrodes whose
# y is below POSTERIOR_FRACTION of the most-posterior y, so the rule scales
# with the montage instead of hardcoding electrode names or a fixed metre cut.
POSTERIOR_FRACTION = 0.5

ALPHA_BAND      = (8.0, 13.0)         # Hz, posterior alpha
RANDOM_STATE    = 42                  # reproducible ICA whitening / fits
ICA_MAX_ITER    = 2000

# --------------------------- Dataset root -----------------------------------
def find_dataset_root(start=None):
    """Locate the dataset root containing sub-* directories.

    Resolution order:
      1. HBN_DATA_ROOT environment variable, if set.
      2. Walk up from the current working directory; at each level accept it if
         it directly contains sub-* dirs, else a single child that does.
    Raises FileNotFoundError if nothing is found, instead of silently using ".".
    """
    env = os.environ.get("HBN_DATA_ROOT")
    if env:
        return Path(env).expanduser().resolve()

    start = Path(start or Path.cwd()).resolve()

    def has_subjects(d):
        try:
            return any(p.is_dir() and p.name.startswith("sub-") for p in d.iterdir())
        except (PermissionError, OSError):
            return False

    for base in [start, *start.parents]:
        if has_subjects(base):
            return base
        try:
            for child in base.iterdir():
                if child.is_dir() and has_subjects(child):
                    return child
        except (PermissionError, OSError):
            pass

    raise FileNotFoundError(
        "Could not locate a dataset root containing 'sub-*' directories. "
        "Set the HBN_DATA_ROOT environment variable to the dataset path."
    )


def subject_bdf_path(root, subject, task=TASK):
    return root / subject / "eeg" / f"{subject}_task-{task}_eeg.bdf"


DATA_ROOT = find_dataset_root()

subject_dirs = sorted(
    p for p in DATA_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("sub-")
)
if not subject_dirs:
    raise FileNotFoundError(f"No sub-* directories under {DATA_ROOT}")

# Validation subject = first subject (alphabetical) that actually has the task
# file, rather than blindly subject_dirs[0].
SUBJECT = None
for d in subject_dirs:
    if subject_bdf_path(DATA_ROOT, d.name).exists():
        SUBJECT = d.name
        break
if SUBJECT is None:
    raise FileNotFoundError(
        f"No subject has a task-{TASK} BDF file under {DATA_ROOT}"
    )

print("CONFIGURATION")
print(f"DATA_ROOT        : {DATA_ROOT}")
print(f"Subjects found   : {len(subject_dirs)}")
print(f"Validation subj  : {SUBJECT}")
print(f"TASK             : {TASK}")
print(f"Band-pass        : {L_FREQ}-{H_FREQ} Hz (Butterworth order {IIR_ORDER}, zero-phase)")
print(f"Epoch duration   : {EPOCH_DURATION} s")

## 2. Load recording and attach montage

The HydroCel-129 montage is attached immediately after loading. A channel-count
assertion guards against an accidental montage / dataset mismatch, which would
otherwise pass silently (`on_missing="warn"`) and corrupt the later
coordinate-based posterior-channel selection.

In [ ]:

bdf_path = subject_bdf_path(DATA_ROOT, SUBJECT)
if not bdf_path.exists():
    raise FileNotFoundError(f"BDF file not found: {bdf_path}")

raw = mne.io.read_raw_bdf(bdf_path, preload=True, verbose=False)

# Guard against a wrong montage / unexpected channel set.
assert len(raw.ch_names) == EXPECTED_NCHAN, (
    f"Expected {EXPECTED_NCHAN} channels for {MONTAGE_NAME}, "
    f"got {len(raw.ch_names)}"
)

raw.set_montage(
    mne.channels.make_standard_montage(MONTAGE_NAME),
    on_missing="warn",
    verbose=False,
)

print(f"Loaded: {bdf_path.name}")
print(raw.info)

## 3. Dataset inspection

In [ ]:

print("DATASET INSPECTION")
sfreq = raw.info["sfreq"]
duration = raw.n_times / sfreq

print(f"Channels           : {len(raw.ch_names)}")
print(f"Sampling frequency : {sfreq:.1f} Hz")
print(f"Duration           : {duration:.1f} s")
print(f"Samples            : {raw.n_times}")

if sfreq != SFREQ_EXPECTED:
    print(f"NOTE: sampling rate {sfreq} Hz != documented {SFREQ_EXPECTED} Hz")

print("\nFirst / last channels:", raw.ch_names[:5], "...", raw.ch_names[-3:])

## 4. Band-pass filter (1-40 Hz, zero-phase Butterworth)

A cleaned copy `raw_clean` is filtered while the pristine `raw` is preserved for
montage/geometry inspection. MNE applies the IIR filter with `filtfilt`, so the
filter is **zero-phase** (no phase distortion) and the effective order is roughly
double the design order.

*Robbins et al. (2020):* cross-pipeline differences are largest in the low
frequencies; the 1 Hz high-pass and band edges are the main levers there, so
delta-band results downstream should be treated as preprocessing-sensitive.

In [ ]:

raw_clean = raw.copy()
raw_clean.filter(
    l_freq=L_FREQ,
    h_freq=H_FREQ,
    method="iir",
    iir_params={"order": IIR_ORDER, "ftype": "butter"},
    verbose=False,
)
print(f"Band-pass applied: {L_FREQ}-{H_FREQ} Hz "
      f"(Butterworth order {IIR_ORDER}, zero-phase via filtfilt)")

## 5. Bad-channel detection (robust, variance-based)

Channels are flagged by **log-variance** using a robust centre/scale
(median and 1.4826*MAD) rather than mean +- 3*SD. The plain mean/SD are inflated
by the very outliers they are meant to catch, so one extreme channel can mask a
second; the robust statistic is far more stable across subjects. Detection runs
**before** the average reference so a bad channel does not leak into the
reference (Kim et al. 2023; Robbins et al. 2020). *Small code adjustment* vs. a
mean+-3*SD rule.

In [ ]:

print("BAD CHANNEL DETECTION (robust log-variance)")

data = raw_clean.get_data()
log_var = np.log(np.var(data, axis=1) + 1e-30)

med = np.median(log_var)
mad = np.median(np.abs(log_var - med))
robust_sd = 1.4826 * mad if mad > 0 else np.std(log_var)

z = (log_var - med) / robust_sd
bad_channels = [raw_clean.ch_names[i] for i in np.where(np.abs(z) > BAD_CH_MAD_THRESH)[0]]

print(f"Median log-var   : {med:.3f}")
print(f"Robust SD        : {robust_sd:.3f}")
print(f"Threshold        : +-{BAD_CH_MAD_THRESH} robust SD")
print(f"Bad channels     : {len(bad_channels)} -> {bad_channels}")

## 6. Interpolate bad channels (spherical spline)

Spherical-spline interpolation is non-linear, so each interpolated channel is
**not** an exact linear combination of the others; this lowers the effective
rank (Kim et al. 2023). That is accounted for in step 8 before ICA.

In [ ]:

print("BAD CHANNEL INTERPOLATION")
n_interpolated = len(bad_channels)

if bad_channels:
    raw_clean.info["bads"] = bad_channels
    raw_clean.interpolate_bads(reset_bads=True, verbose=False)
    print(f"Interpolated {n_interpolated}: {bad_channels}")
else:
    print("No bad channels to interpolate")

## 7. Average reference

Re-referencing to the average. Note that the average reference removes one
degree of freedom (rank drops by 1). Combined with non-linear interpolation,
this is exactly the rank reduction Kim et al. (2023) warn about; it is measured
explicitly in the next cell. (MNE's average reference does not drop a channel,
so it does not reproduce the EEGLAB "method A" rank-deficiency bug, but the
rank-by-1 reduction is still real and is handled before ICA.)

In [ ]:

raw_clean.set_eeg_reference("average", projection=False, verbose=False)
print("Average reference applied")

## 8. Effective data rank (ghost-IC guard)

**Kim et al. (2023), key fix.** ICA must not be asked for more components than
the data rank, or it emits *ghost ICs* (flat-PSD, no-ERP components with
deceptively normal topographies). We count eigenvalues of the channel
covariance above a **relative** tolerance (units-invariant, unlike an absolute
1e-7) and limit ICA to that rank in step 10. The empirical rank is cross-checked
against the theoretical value `n_channels - n_interpolated - 1` (the -1 is the
average reference).

In [ ]:

print("EFFECTIVE RANK CHECK")

dat = raw_clean.get_data()
eigvals = np.linalg.eigvalsh(np.cov(dat))
eigvals = eigvals[eigvals > 0]
n_rank = int(np.sum(eigvals > eigvals.max() * RANK_REL_TOL))

theoretical = len(raw_clean.ch_names) - n_interpolated - 1  # -1 = avg ref

print(f"Channels             : {len(raw_clean.ch_names)}")
print(f"Interpolated         : {n_interpolated}")
print(f"Theoretical rank     : {theoretical}  (n_chan - n_interp - 1)")
print(f"Empirical rank       : {n_rank}  (eigvals > {RANK_REL_TOL} * max)")

if n_rank != theoretical:
    print("NOTE: empirical != theoretical; ICA will use the empirical rank.")

## 9. Posterior electrodes from montage coordinates

Posterior electrodes (for alpha validation) are chosen from montage geometry,
not channel names. In MNE head coordinates (metres, nasion-relative RAS) y < 0
is posterior. The cut is expressed as a fraction of the most-posterior y, so it
scales with the montage rather than hardcoding a fixed metre value.

In [ ]:

print("POSTERIOR CHANNEL IDENTIFICATION")

montage = raw.get_montage()
if montage is None:
    raise RuntimeError("No montage; cannot identify posterior channels.")

positions = montage.get_positions()["ch_pos"]
ys = {ch: positions[ch][1] for ch in raw.ch_names if ch in positions}
min_y = min(ys.values())
y_threshold = POSTERIOR_FRACTION * min_y  # min_y < 0, so threshold < 0

posterior = [ch for ch, y in ys.items() if y < y_threshold]

print(f"Most-posterior y     : {min_y:.3f} m")
print(f"y threshold          : {y_threshold:.3f} m  ({POSTERIOR_FRACTION} * min_y)")
print(f"Posterior channels   : {len(posterior)}")
print(posterior)

## 10. ICA (rank-limited, manual inspection)

ICA is fit on this single validation subject only. `n_components` is set to the
**effective rank** from step 8 -- this is the PCA-for-rank-adjustment that
prevents ghost ICs (Kim et al. 2023). FastICA is kept to match the existing
pipeline; extended-infomax / picard are common alternatives (Kim et al. used
extended infomax) and would be a drop-in change if desired.

Component selection is **manual**: inspect the plots, then edit the exclusion
list in the next cell.

In [ ]:

print(f"Fitting ICA with n_components = {n_rank} (effective rank)...")

ica = mne.preprocessing.ICA(
    n_components=n_rank,          # rank-limited -> no ghost ICs
    method="fastica",
    random_state=RANDOM_STATE,    # reproducible
    max_iter=ICA_MAX_ITER,
)
ica.fit(raw_clean)

print(f"Converged in {ica.n_iter_} iters (max {ICA_MAX_ITER}): "
      f"{ica.n_iter_ < ICA_MAX_ITER}")
print(f"Components: {ica.n_components_}")

# Topographies for manual inspection.
ica.plot_components(show=False)
plt.show()

# Continuous source time courses.
ica.plot_sources(raw_clean, start=60, stop=120,
                 title="ICA sources (t=60-120 s)", show=False)
plt.show()

# Detailed properties of the first few components (topography + PSD + ERP-image).
# A FLAT PSD with NO temporal structure but a normal-looking topography is the
# ghost-IC signature (Kim et al. 2023) -- check the PSD, not just the map.
n_show = min(10, ica.n_components_)
ica.plot_properties(raw_clean, picks=list(range(n_show)))
plt.show()

### Manual component rejection

Set `ica.exclude` from your inspection above. The list below is an
**example for this validation subject only** -- component indices are
subject-specific and are NOT valid for any other subject. Leave it empty to
reject nothing.

In [ ]:

# ---------------------------------------------------------------------------
# EXAMPLE exclusion list for the validation subject ONLY.
# Replace with the indices you identified from the plots above.
# These indices are NOT universally valid and are NOT used in the batch stage.
# ---------------------------------------------------------------------------
ica.exclude = []   # e.g. [0, 3, 7] after inspecting THIS subject

print(f"Components marked for removal: {ica.exclude}")

raw_ica = raw_clean.copy()
if ica.exclude:
    ica.apply(raw_ica)
    print(f"Removed {len(ica.exclude)} component(s)")

    removed = raw_clean.get_data() - raw_ica.get_data()
    removed_rms  = np.sqrt(np.mean(removed ** 2))
    original_rms = np.sqrt(np.mean(raw_clean.get_data() ** 2))
    print(f"Removed RMS / original RMS = {100 * removed_rms / original_rms:.2f}%")
else:
    print("No components removed (raw_ica == raw_clean).")

## 11. Alpha-preservation validation

Sanity check that ICA did not eat posterior alpha. The **authoritative** metric
is integrated alpha-band power (the band integral is more robust than a single
peak value); the peak frequency is reported alongside for context.

In [ ]:

from scipy.integrate import simpson

psd_before = raw_clean.compute_psd(fmin=L_FREQ, fmax=H_FREQ, picks=posterior)
psd_after  = raw_ica.compute_psd(fmin=L_FREQ, fmax=H_FREQ, picks=posterior)

freqs  = psd_before.freqs
before = psd_before.get_data().mean(axis=0)
after  = psd_after.get_data().mean(axis=0)

alpha_idx = np.where((freqs >= ALPHA_BAND[0]) & (freqs <= ALPHA_BAND[1]))[0]

# Authoritative metric: integrated band power.
power_before = simpson(before[alpha_idx], x=freqs[alpha_idx])
power_after  = simpson(after[alpha_idx],  x=freqs[alpha_idx])
power_change = 100 * (power_after - power_before) / power_before

# Context: peak frequency.
peak_before = freqs[alpha_idx[np.argmax(before[alpha_idx])]]
peak_after  = freqs[alpha_idx[np.argmax(after[alpha_idx])]]

print("POSTERIOR ALPHA PRESERVATION")
print(f"Posterior channels   : {len(posterior)}")
print(f"Alpha band power      before={power_before:.3e}  after={power_after:.3e}")
print(f"Alpha band power change (authoritative): {power_change:.2f}%")
print(f"Alpha peak freq       before={peak_before:.2f} Hz  after={peak_after:.2f} Hz")

if power_change > -3:
    print("Alpha preserved within tolerance.")
else:
    print("Notable alpha reduction -- re-check excluded components.")

plt.figure(figsize=(8, 5))
plt.plot(freqs, before, label="Before ICA")
plt.plot(freqs, after, label="After ICA")
plt.axvspan(ALPHA_BAND[0], ALPHA_BAND[1], alpha=0.2, label="Alpha")
plt.xlim(L_FREQ, H_FREQ)
plt.xlabel("Frequency (Hz)"); plt.ylabel("PSD")
plt.title("Posterior alpha preservation"); plt.legend(); plt.tight_layout()
plt.show()

## 12. Fixed-length epochs (single subject)

Validation-stage epoching of the ICA-cleaned data.

**Brookshire et al. (2024):** these per-subject segments must, in any downstream
model, be split **by subject** (e.g. GroupKFold) -- never pooled and split by
epoch, and never with each channel as its own observation. Either shortcut leaks
subject identity and inflates accuracy.

In [ ]:

epochs = mne.make_fixed_length_epochs(
    raw_ica, duration=EPOCH_DURATION, overlap=0.0, preload=True, verbose=False
)
print(f"Epochs            : {len(epochs)}")
print(f"Duration          : {EPOCH_DURATION} s")
print(f"Samples/epoch     : {epochs.get_data().shape[-1]}")

## 13. Events file inspection

In [ ]:

events_tsv = DATA_ROOT / SUBJECT / "eeg" / f"{SUBJECT}_task-{TASK}_events.tsv"
print("EVENTS FILE INSPECTION")

if events_tsv.exists():
    ev = pd.read_csv(events_tsv, sep="\t")
    print(f"Found {events_tsv.name}: {len(ev)} events, columns={list(ev.columns)}")
    print(ev.head().to_string(index=False))
else:
    print("No events.tsv (normal for some RestingState recordings).")

## 14. Quality-table inspection

In [ ]:

quality_tsv = DATA_ROOT / "code" / f"{TASK}_quality_table.tsv"
print("QUALITY TABLE INSPECTION")

if quality_tsv.exists():
    qdf = pd.read_csv(quality_tsv, sep=None, engine="python")
    subj_id = SUBJECT.replace("sub-", "")
    subject_col = qdf.columns[0]
    row = qdf[qdf[subject_col].astype(str).str.strip() == subj_id]
    print(f"Found {quality_tsv.name}: {len(qdf)} subjects listed")
    if not row.empty:
        print(row.to_string(index=False))
    else:
        print(f"{SUBJECT} not in quality table")
else:
    print("Quality table not found (expected in some mini releases).")

In [ ]:
import json
import sys
import platform
import datetime

output_dir = DATA_ROOT / "preprocessed_epochs"
output_dir.mkdir(exist_ok=True)

MIN_EPOCHS = 30   # flag subjects with too little surviving data

results = []
print(f"Processing {len(subject_dirs)} subjects (ICA excluded by design)...\n")

for subj_dir in subject_dirs:
    subj_id = subj_dir.name
    path = subject_bdf_path(DATA_ROOT, subj_id)

    if not path.exists():
        results.append({"subject": subj_id, "status": "NO_FILE"})
        continue

    try:
        r = mne.io.read_raw_bdf(path, preload=True, verbose=False)
        if len(r.ch_names) != EXPECTED_NCHAN:
            results.append({"subject": subj_id, "status": "BAD_NCHAN"})
            continue

        # Hard sampling-rate guard: refuse to proceed if the recording does
        # not match the documented rate, because every downstream PSD /
        # FOOOF resolution depends on it (audit W2).
        if abs(r.info["sfreq"] - SFREQ_EXPECTED) > 1e-6:
            raise ValueError(
                f"sfreq={r.info['sfreq']} Hz != expected {SFREQ_EXPECTED} Hz"
            )

        r.set_montage(mne.channels.make_standard_montage(MONTAGE_NAME),
                      on_missing="warn", verbose=False)
        r.filter(l_freq=L_FREQ, h_freq=H_FREQ, method="iir",
                 iir_params={"order": IIR_ORDER, "ftype": "butter"}, verbose=False)

        # Robust bad-channel detection (same rule as the single-subject cell),
        # BEFORE the average reference.
        d = r.get_data()
        lv = np.log(np.var(d, axis=1) + 1e-30)
        m = np.median(lv)
        rsd = 1.4826 * np.median(np.abs(lv - m))
        rsd = rsd if rsd > 0 else np.std(lv)
        bads = [r.ch_names[i] for i in np.where(np.abs((lv - m) / rsd) > BAD_CH_MAD_THRESH)[0]]
        if bads:
            r.info["bads"] = bads
            r.interpolate_bads(reset_bads=True, verbose=False)

        r.set_eeg_reference("average", projection=False, verbose=False)

        ep = mne.make_fixed_length_epochs(
            r, duration=EPOCH_DURATION, overlap=0.0, preload=True, verbose=False
        )

        out_file = output_dir / f"{subj_id}_task-{TASK}_clean_epo.fif"
        ep.save(out_file, overwrite=True, verbose=False)

        results.append({
            "subject": subj_id,
            "n_epochs": len(ep),
            "bad_channels": len(bads),
            "low_epochs": len(ep) < MIN_EPOCHS,
            "status": "OK",
        })
        print(f"{subj_id}: {len(ep)} epochs | {len(bads)} bad ch"
              + ("  [LOW EPOCHS]" if len(ep) < MIN_EPOCHS else ""))

    except Exception as e:
        results.append({"subject": subj_id, "status": f"ERROR: {str(e)[:60]}"})
        print(f"{subj_id}: ERROR -- {str(e)[:60]}")

results_df = pd.DataFrame(results)
summary_file = output_dir / "preprocessing_summary.csv"
results_df.to_csv(summary_file, index=False)

# ---------- provenance.json (audit W8) ----------
# Records *how* the saved data was produced: package versions, every
# preprocessing constant, and a UTC timestamp. Downstream stages can read
# this without re-running the pipeline.
try:
    import scipy
    scipy_ver = scipy.__version__
except Exception:
    scipy_ver = "unavailable"

provenance = {
    "timestamp_utc": datetime.datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "mne": mne.__version__,
    "numpy": np.__version__,
    "scipy": scipy_ver,
    "pandas": pd.__version__,
    "constants": {
        "TASK": TASK,
        "MONTAGE_NAME": MONTAGE_NAME,
        "EXPECTED_NCHAN": EXPECTED_NCHAN,
        "SFREQ_EXPECTED": SFREQ_EXPECTED,
        "L_FREQ": L_FREQ,
        "H_FREQ": H_FREQ,
        "IIR_ORDER": IIR_ORDER,
        "BAD_CH_MAD_THRESH": BAD_CH_MAD_THRESH,
        "EPOCH_DURATION": EPOCH_DURATION,
        "RANK_REL_TOL": RANK_REL_TOL,
        "POSTERIOR_FRACTION": POSTERIOR_FRACTION,
        "ALPHA_BAND": list(ALPHA_BAND),
        "RANDOM_STATE": RANDOM_STATE,
        "ICA_MAX_ITER": ICA_MAX_ITER,
        "MIN_EPOCHS": MIN_EPOCHS,
    },
    "ica_in_batch": False,
    "n_subjects_attempted": len(subject_dirs),
    "n_subjects_ok": int((results_df["status"] == "OK").sum()) if "status" in results_df.columns else 0,
}
provenance_file = output_dir / "provenance.json"
provenance_file.write_text(json.dumps(provenance, indent=2), encoding="utf-8")

print("\n" + "=" * 60)
print("BATCH COMPLETE")
print("=" * 60)
print(f"Output dir   : {output_dir}")
print(f"Summary file : {summary_file}")
print(f"Provenance   : {provenance_file}")
print("\nNOTE (Brookshire et al. 2024): split these epochs BY SUBJECT "
      "(e.g. GroupKFold) downstream, never by pooled epoch.")
print("\n" + results_df.to_string(index=False))

In [ ]:

output_dir = DATA_ROOT / "preprocessed_epochs"
output_dir.mkdir(exist_ok=True)

MIN_EPOCHS = 30   # flag subjects with too little surviving data

results = []
print(f"Processing {len(subject_dirs)} subjects (ICA excluded by design)...\n")

for subj_dir in subject_dirs:
    subj_id = subj_dir.name
    path = subject_bdf_path(DATA_ROOT, subj_id)

    if not path.exists():
        results.append({"subject": subj_id, "status": "NO_FILE"})
        continue

    try:
        r = mne.io.read_raw_bdf(path, preload=True, verbose=False)
        if len(r.ch_names) != EXPECTED_NCHAN:
            results.append({"subject": subj_id, "status": "BAD_NCHAN"})
            continue

        r.set_montage(mne.channels.make_standard_montage(MONTAGE_NAME),
                      on_missing="warn", verbose=False)
        r.filter(l_freq=L_FREQ, h_freq=H_FREQ, method="iir",
                 iir_params={"order": IIR_ORDER, "ftype": "butter"}, verbose=False)

        # Robust bad-channel detection (same rule as the single-subject cell),
        # BEFORE the average reference.
        d = r.get_data()
        lv = np.log(np.var(d, axis=1) + 1e-30)
        m = np.median(lv)
        rsd = 1.4826 * np.median(np.abs(lv - m))
        rsd = rsd if rsd > 0 else np.std(lv)
        bads = [r.ch_names[i] for i in np.where(np.abs((lv - m) / rsd) > BAD_CH_MAD_THRESH)[0]]
        if bads:
            r.info["bads"] = bads
            r.interpolate_bads(reset_bads=True, verbose=False)

        r.set_eeg_reference("average", projection=False, verbose=False)

        ep = mne.make_fixed_length_epochs(
            r, duration=EPOCH_DURATION, overlap=0.0, preload=True, verbose=False
        )

        out_file = output_dir / f"{subj_id}_task-{TASK}_clean_epo.fif"
        ep.save(out_file, overwrite=True, verbose=False)

        results.append({
            "subject": subj_id,
            "n_epochs": len(ep),
            "bad_channels": len(bads),
            "low_epochs": len(ep) < MIN_EPOCHS,
            "status": "OK",
        })
        print(f"{subj_id}: {len(ep)} epochs | {len(bads)} bad ch"
              + ("  [LOW EPOCHS]" if len(ep) < MIN_EPOCHS else ""))

    except Exception as e:
        results.append({"subject": subj_id, "status": f"ERROR: {str(e)[:60]}"})
        print(f"{subj_id}: ERROR -- {str(e)[:60]}")

results_df = pd.DataFrame(results)
summary_file = output_dir / "preprocessing_summary.csv"
results_df.to_csv(summary_file, index=False)

print("\n" + "=" * 60)
print("BATCH COMPLETE")
print("=" * 60)
print(f"Output dir   : {output_dir}")
print(f"Summary file : {summary_file}")
print("\nNOTE (Brookshire et al. 2024): split these epochs BY SUBJECT "
      "(e.g. GroupKFold) downstream, never by pooled epoch.")
print("\n" + results_df.to_string(index=False))